Homework 4 - LSTM

Goal: Build a Long Short Term Memory (LSTM) neural network to generate music by generating sequence of notes in a music. (10 points)

Data: A classical piano composition by Mozart is used to train the neural network. Use the sample MIDI (Musical Instrument Digital Interface) file, mozart.mid. A MIDI music file consists of several notes that have attributes like start time, end time, pitch and velocity. The python package pretty_midi could be used to extract this information from the music file.

Pre-defined functions: The provided functions below generate input data from MIDI files, generate new sequence of notes from a given sequence, and to create a MIDI file from the generated sequence.

1. Historical input data of a particular sequence length along with the corresponding output can be generated using the function generate_input_data. The converter parses the MIDI file and generates a list of all notes and chords in the file. Each chord is appended by encoding the ID of every note in the chord together into a single string. These encodings allow the decoding of the output generated by the LSTM network into correct notes and chords. Once the sequential list of all notes and chords is available, historical input data is generated. Finally, the network input data is normalized and the network output data is converted into a categorical array.
2. The function generate_sequence could be used to generate a new sequence.
3. The function create_midi will convert any given sequence of notes into a MIDI file.

In [ ]:
# Import necessary libraries
from music21 import converter, instrument, note, chord, stream
import tensorflow, pretty_midi

import shutil
from google.colab import files
import os


from pathlib import Path
from dotenv import load_dotenv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import random_split, DataLoader, TensorDataset
from IPython.display import display, Image
import time
import copy
from datetime import datetime

# Loss plots and deliverable images are saved under this folder (single source of truth)
PLOT_OUTPUT_DIR = "Plot JPGs"

In [8]:
def generate_input_data(song_path):
    
    notes = []
    midi = converter.parse(song_path)
    print("Parsing %s" % song_path)

    # use flat notes only
    notes_to_parse = midi.flat.notes

    for element in notes_to_parse:
        if isinstance(element, note.Note):
            notes.append(str(element.pitch))
        elif isinstance(element, chord.Chord):
            notes.append('.'.join(str(n) for n in element.normalOrder))

    sequence_length = 100

    # get all pitch names
    pitchnames = sorted(set(item for item in notes))

     # create a dictionary to map pitches to integers
    note_to_int = dict((note, number) for number, note in enumerate(pitchnames))

    network_input = []
    network_output = []

    # create input sequences and the corresponding outputs
    for i in range(0, len(notes) - sequence_length, 1):
        sequence_in = notes[i:i + sequence_length]
        sequence_out = notes[i + sequence_length]
        network_input.append([note_to_int[char] for char in sequence_in])
        network_output.append(note_to_int[sequence_out])

    n_patterns = len(network_input)

    # reshape the input into a format compatible with LSTM layers
    network_input = np.reshape(network_input, (n_patterns, sequence_length, 1))
    # normalize input
    n_vocab = len(set(notes))
    network_input = network_input / float(n_vocab)

    # Class indices for PyTorch CrossEntropyLoss (replaces Keras to_categorical)
    network_output = np.array(network_output, dtype=np.int64)

    return network_input, network_output, pitchnames, n_vocab

In [3]:
def generate_sequence(model, network_input, pitchnames, n_vocab, device=None):
    """Generate notes using a trained PyTorch model (same sliding-window logic as the Keras recipe)."""
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # pick the last sequence from the input as a starting point for the prediction
    start = -1

    int_to_note = dict((number, note) for number, note in enumerate(pitchnames))

    pattern = network_input[start]
    prediction_output = []

    model.eval()
    with torch.no_grad():
        # generate 500 notes
        for _ in range(500):
            prediction_input = np.reshape(pattern, (1, len(pattern), 1))
            prediction_input = prediction_input / float(n_vocab)

            x = torch.FloatTensor(prediction_input).to(device)
            logits = model(x)
            index = int(torch.argmax(logits, dim=1).item())
            result = int_to_note[index]
            prediction_output.append(result)

            pattern = np.append(pattern, index)
            pattern = pattern[1 : len(pattern)]

    return prediction_output

In [ ]:
def create_midi(prediction_output, output_path="test_output.mid"):
    """Convert the output from the prediction to notes and create a MIDI file."""
    offset = 0
    output_notes = []

    # create note and chord objects based on the values generated by the model
    for pattern in prediction_output:
        # pattern is a chord
        if ('.' in pattern) or pattern.isdigit():
            notes_in_chord = pattern.split('.')
            notes = []
            for current_note in notes_in_chord:
                new_note = note.Note(int(current_note))
                new_note.storedInstrument = instrument.Piano()
                notes.append(new_note)
            new_chord = chord.Chord(notes)
            new_chord.offset = offset
            output_notes.append(new_chord)
        # pattern is a note
        else:
            new_note = note.Note(pattern)
            new_note.offset = offset
            new_note.storedInstrument = instrument.Piano()
            output_notes.append(new_note)

        # increase offset each iteration so that notes do not stack
        offset += 0.5

    midi_stream = stream.Stream(output_notes)

    midi_stream.write("midi", fp=output_path)

In [ ]:
## Import data

# Locate mozart.mid (repo root or HW_04_LSTM/)
if os.path.exists("mozart.mid"):
    file_path = "mozart.mid"
elif os.path.exists(os.path.join("HW_04_LSTM", "mozart.mid")):
    file_path = os.path.join("HW_04_LSTM", "mozart.mid")
else:
    file_path = "mozart.mid"

network_input, network_output, pitchnames, n_vocab = generate_input_data(file_path)
num_classes = len(pitchnames)
SEQUENCE_LENGTH = network_input.shape[1]

# These arrays are the supervised training signal: input windows + next-note labels from the MIDI chop-up.
assert network_input.shape[0] == len(network_output)

Architecture: Define a Sequential model, wherein the layers are stacked sequentially and each layer has exactly one input tensor and one output tensor.  Please build a LSTM neural network, in the form of a sequential model, by adding the layers to the model using the configuration below.

·       LSTM | Units : 512 | Recurrent dropout : 0.3 | Return sequences : True

·       LSTM | Units : 512 | Recurrent dropout : 0.3 | Return sequences : True

·       Flatten


·       Dense | Units : 256 | Activation : ReLU

·       Dense | Units : 95 (# total notes) | Activation : Softmax

In [ ]:
# Define pytorch model class (Sequential-style stack: 2x LSTM -> Flatten -> Dense -> Dense)
# Keras "recurrent dropout" is approximated with dropout between stacked LSTM layers (PyTorch nn.LSTM).


class MusicLSTM(nn.Module):
    def __init__(
        self,
        sequence_length=100,
        num_classes=95,
        hidden_size=512,
        lstm_dropout=0.3,
    ):
        super().__init__()
        self.sequence_length = sequence_length
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_size,
            num_layers=2,
            batch_first=True,
            dropout=lstm_dropout,
        )
        self.flatten = nn.Flatten()
        flat_dim = sequence_length * hidden_size
        self.fc1 = nn.Linear(flat_dim, 256)
        self.relu = nn.ReLU()
        # Logits for nn.CrossEntropyLoss (softmax is inside the loss)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.flatten(out)
        out = self.relu(self.fc1(out))
        out = self.fc2(out)
        return out


In [ ]:

def train(model, loader, criterion, optimizer, device):
    """Train the model for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def evaluate(model, loader, criterion, device):
    """Evaluate the model on validation or test data."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = correct / total
    return epoch_loss, epoch_acc


def train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=50):
    """Train and record training / validation loss and accuracy each epoch."""
    training_loss_curve = []
    validation_loss_curve = []
    training_accuracy = []
    validation_accuracy = []

    for epoch in range(num_epochs):
        train_loss, train_acc = train(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        training_loss_curve.append(train_loss)
        training_accuracy.append(train_acc)
        validation_loss_curve.append(val_loss)
        validation_accuracy.append(val_acc)

        timestamp = datetime.now().strftime("%H:%M:%S")
        print(
            f"[{timestamp}] Epoch {epoch+1}/{num_epochs} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

    return training_loss_curve, validation_loss_curve, training_accuracy, validation_accuracy


# PLOT LOSS CURVES
def plot_loss(curve, dataset, model_name, output_dir="Plot JPGs"):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    plt.figure()
    plt.plot(curve)
    plt.xlabel("Epoch")
    if dataset == "Training":
        plt.ylabel("Training Loss")
    elif dataset == "Validation":
        plt.ylabel("Validation Loss")

    plt.title(
        f"Categorical Cross Entropy Loss Curve for {dataset} Dataset, Model: {model_name}"
    )
    filename = f"{output_dir}/LossCurve_{dataset}_Dataset_Model_{model_name}.jpg"

    plt.savefig(filename)
    plt.close()

Training: The sequential model with the above configuration is finally compiled with the ADAM optimizer to minimize the categorical cross entropy loss. The model is then trained using network_input and network_output as input and target data for 50 epochs while dividing the data into a batch size of 256. Finally, the trained model can be used to generate new sequences from a given sequence of notes using the pre-defined functions introduced above.

In [ ]:
# Experiment Function
def run_experiment(model, model_name, train_loader, val_loader, test_loader, max_epochs=50):
    """Train with Adam + categorical cross-entropy, plot loss curves, return a results row."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()))

    training_loss_curve, validation_loss_curve, training_accuracy, validation_accuracy = train_model(
        model, train_loader, val_loader, criterion, optimizer, device, max_epochs
    )
    print("Training complete")

    final_test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    print("Evaluation complete")

    plot_loss(training_loss_curve, "Training", model_name, output_dir=PLOT_OUTPUT_DIR)
    plot_loss(validation_loss_curve, "Validation", model_name, output_dir=PLOT_OUTPUT_DIR)
    print("Plots created")

    row = {
        "Model Name": model_name,
        "Epochs": max_epochs,
        "Training Accuracy (%)": training_accuracy[-1] * 100.0,
        "Validation Accuracy (%)": validation_accuracy[-1] * 100.0,
        "Test Accuracy (%)": test_acc * 100.0,
        "Final Test Loss": final_test_loss,
    }
    return row

In [ ]:
# Hyperparameters
MAX_EPOCHS = 50
BATCH_SIZE = 256
SHUFFLE_SEED = 42
torch.manual_seed(SHUFFLE_SEED)

MODEL_NAME = "LSTM_Music"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Wrap the MIDI chop-up in tensors: each batch is (windows from network_input, labels from network_output).
X_tensor = torch.FloatTensor(network_input)
y_tensor = torch.LongTensor(network_output)
assert X_tensor.shape[0] == y_tensor.shape[0] == network_input.shape[0] == len(network_output)
dataset = TensorDataset(X_tensor, y_tensor)
print(
    f"Training data: {len(dataset)} examples from generate_input_data; "
    f"input {tuple(network_input.shape)}, targets {network_output.shape}"
)

total_n = len(dataset)
val_size = int(0.15 * total_n)
train_size = total_n - val_size
train_subset, val_subset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SHUFFLE_SEED),
)

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    worker_init_fn=lambda worker_id: np.random.seed(SHUFFLE_SEED),
)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = val_loader

model = MusicLSTM(
    sequence_length=SEQUENCE_LENGTH,
    num_classes=num_classes,
    hidden_size=512,
    lstm_dropout=0.3,
)

timestamp = datetime.now().strftime("%H:%M:%S")
print(f"[{timestamp}] Run Start")

results_row = run_experiment(
    model, MODEL_NAME, train_loader, val_loader, test_loader, max_epochs=MAX_EPOCHS
)
results_df = pd.DataFrame([results_row])
display(results_df)

timestamp = datetime.now().strftime("%H:%M:%S")
print(f"[{timestamp}] Run Finish")

# Zip and download loss plots (Colab-friendly)
plot_zip_dir = PLOT_OUTPUT_DIR
if os.path.isdir(plot_zip_dir):
    shutil.make_archive("Plot_JPGs", "zip", plot_zip_dir)
    print("Created Plot_JPGs.zip")
    if files is not None:
        try:
            files.download("Plot_JPGs.zip")
        except Exception as e:
            print(f"Download skipped or failed: {e}")
else:
    print(f"Plot folder not found: {plot_zip_dir}")


In [ ]:
# Generate new sequence of 500 notes (uses generate_sequence + create_midi helpers)
os.makedirs("final_midi", exist_ok=True)
midi_out = os.path.join("final_midi", "generated.mid")

timestamp = datetime.now().strftime("%H:%M:%S")
print(f"[{timestamp}] Generate Start")

prediction_output = generate_sequence(
    model, network_input, pitchnames, n_vocab, device=device
)
create_midi(prediction_output, output_path=midi_out)
print(f"Saved: {midi_out}")

timestamp = datetime.now().strftime("%H:%M:%S")
print(f"[{timestamp}] Generate Finish")

if os.path.isdir("final_midi"):
    shutil.make_archive("final_midi", "zip", "final_midi")
    print("Created final_midi.zip")
    if files is not None:
        try:
            files.download("final_midi.zip")
        except Exception as e:
            print(f"Download skipped or failed: {e}")

Deliverables: Plot the loss curves for the LSTM model using the loss recorded at each step of the training process. Also, upload a sample MIDI file newly generated using the trained LSTM model containing a notes sequence of length 500. Please make sure to submit your working code files along with the generated sample MIDI file and the plots. 

In [ ]:
# Deliverable: show every plot image saved under the plot folder
from IPython.display import Image as IPyImage

_plot_dir = Path(PLOT_OUTPUT_DIR)
if not _plot_dir.is_dir():
    print(f"No plot folder yet: {_plot_dir} (run training first)")
else:
    _exts = {".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"}
    _paths = sorted(
        p for p in _plot_dir.iterdir() if p.is_file() and p.suffix in _exts
    )
    if not _paths:
        print(f"No image files in {_plot_dir}")
    for _p in _paths:
        print(_p.name)
        display(IPyImage(filename=str(_p)))
